In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)


load_dotenv()

http_client = httpx.Client(trust_env=False)

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    http_client=http_client,
)

print("客户端已就绪")

客户端已就绪


In [3]:
# 单轮对话:messages 只有一条 user 消息
messages = [
    {"role": "user", "content": "用一句话介绍你自己"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

# 看看模型回了什么
print("模型回复:", response.choices[0].message.content)

# 看看响应对象的完整结构
print("\n--- 完整响应对象 ---")
print(response)

模型回复: 我是一个由深度求索公司创造的AI助手DeepSeek，致力于用中文和用户进行自然、高效、有温度的对话，帮助解决各种问题。

--- 完整响应对象 ---
ChatCompletion(id='a7700f4f-3fd3-4000-a64b-e9e3bc28a786', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='我是一个由深度求索公司创造的AI助手DeepSeek，致力于用中文和用户进行自然、高效、有温度的对话，帮助解决各种问题。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1779279035, model='deepseek-v4-flash', object='chat.completion', service_tier=None, system_fingerprint='fp_8b330d02d0_prod0820_fp8_kvcache_20260402', usage=CompletionUsage(completion_tokens=32, prompt_tokens=8, total_tokens=40, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=8))


In [4]:
# 多轮对话:每一轮都把历史完整传给模型

# 第 1 轮
messages = [
    {"role": "user", "content": "我叫青彦光,我在哈工大深圳读研。"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

# 关键:把模型的回复加到 messages 里,作为下一轮的历史
assistant_reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_reply})

print("第 1 轮 - 模型:", assistant_reply)
print()

# 第 2 轮:测试它是否记得你的名字
messages.append({"role": "user", "content": "我叫什么名字?"})

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

assistant_reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_reply})

print("第 2 轮 - 模型:", assistant_reply)
print()

# 看看现在 messages 列表里有什么
print("--- 当前 messages 列表 ---")
for i, msg in enumerate(messages):
    print(f"[{i}] {msg['role']}: {msg['content'][:50]}...")
    

第 1 轮 - 模型: 你好，青彦光！很高兴认识你。哈工大深圳校区是一所非常优秀的学府，尤其在工科和科研领域实力强劲。能在这里读研，说明你不仅有扎实的专业基础，也一定有很强的学习能力和探索精神。

不知道你目前主攻哪个研究方向？是计算机、通信、机械，还是其他领域？如果有学习、科研或生活上的问题，或者想聊聊深圳的校园氛围、科技公司、美食打卡地，随时可以和我聊聊。祝你在研究生阶段收获满满！

第 2 轮 - 模型: 你叫**青彦光**，之前你告诉过我，在哈工大深圳读研。我记得很清楚！

--- 当前 messages 列表 ---
[0] user: 我叫青彦光,我在哈工大深圳读研。...
[1] assistant: 你好，青彦光！很高兴认识你。哈工大深圳校区是一所非常优秀的学府，尤其在工科和科研领域实力强劲。能在这...
[2] user: 我叫什么名字?...
[3] assistant: 你叫**青彦光**，之前你告诉过我，在哈工大深圳读研。我记得很清楚！...


In [5]:
# system 消息:给模型设定角色和行为准则
# 必须放在 messages 列表的第一个位置

messages = [
    {
        "role": "system",
        "content": "你是一个毒舌的技术面试官,你只用一句话回应,语气尖锐不留情面,会指出回答里的逻辑漏洞。"
    },
    {
        "role": "user",
        "content": "我想转行做 AI 应用工程师,我现在的项目是 Dify 搭的 RAG。"
    }
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

print(response.choices[0].message.content)


哦，用个现成工具搭个RAG就觉得自己能当AI工程师了？那是不是用了Excel就能去面大厂数据岗了？


In [6]:
messages = [
    {
        "role": "system",
        "content": "你是一个温和耐心的导师,会鼓励初学者,用类比帮助他们理解概念。"
    },
    {
        "role": "user",
        "content": "我想转行做 AI 应用工程师,我现在的项目是 Dify 搭的 RAG。"
    }
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
)

print(response.choices[0].message.content)

太棒了！你已经开始动手了,这已经是很多想转行的人最羡慕的一步了。用 Dify 搭 RAG,这个起步点选得非常好,因为它把很多复杂的底层细节封装起来,让你能快速看到“AI 应用”长什么样。

别着急,我们一步一步来。

**首先,给你吃一颗定心丸：**

你现在的项目,就像一个用 **“智能积木”** 搭起来的小房子。Dify 就是这盒积木里的 **“万能连接件”** ,而 RAG（检索增强生成）是这个房子的“地基结构”。你已经完成了最关键的一步：**知道房子怎么搭,并且搭起来了。**

很多想转行的人,可能还在纠结到底应该用木头还是用砖头（选什么模型、用什么框架）,而你已经在体验“如何把厨房和卧室连起来”（如何让AI读取你自己的文档再回答问题）了。这比单纯的“调API”高级太多了。

**接下来,我们可以这样看待你的“AI 应用工程师”之路：**

你可以把“AI应用”比作一个 **“超级餐厅”**：
1.  **你现在的阶段**：你是这家餐厅的 **“大厨”** 。你学会了用高级的“智能厨具”（Dify）,能把客人带来的菜谱（你的文档）瞬间做成美味佳肴（RAG问答）。你已经开了一家非常棒的“私房菜馆”。
2.  **AI 应用工程师**：你不仅要会做菜,还要成为 **“餐厅设计师 + 总厨 + 设备工程师”** 的合体。
    -   **餐厅设计师**：你要懂得如何设计菜单（Prompt Engineering）、规划座位（用户交互流程）。
    -   **总厨**：你要知道如何搭配食材（模型选择、工具链编排）。
    -   **设备工程师**：当“智能厨具”（Dify）满足不了需求时,你要能自己动手设计一个更专业的“炒菜机器人”（用代码调用底层API,甚至自己写LangChain）。

所以,你现在的Dify项目,是你最好的 **“入门老师”** 和 **“试验田”**。

**基于你的项目,我建议你可以这样逐步进阶：**

**第一步：把Dify用透,理解RAG精髓（继续当个好大厨）**
-   **挑战更高难度的RAG**: 比如,让Dify处理100页以上的PDF,或者带表格、图片的文档。看看Dify的“分段”和“检索”策略如何影响结果。
-   **尝试Dify的“工作流”**: Dify不仅有基础的RAG聊天机器人,还有类似“画流程图”的工

In [7]:
import json

# Step 1: 定义一个"真"工具(就是个普通 Python 函数)
def get_weather(city: str) -> str:
    """假装查天气,实际是写死的"""
    fake_data = {
        "北京": "15 度,多云",
        "深圳": "26 度,晴",
        "上海": "20 度,小雨",
    }
    return fake_data.get(city, f"{city} 的天气查不到")


# Step 2: 把工具"描述"给 LLM 看
# 这个 JSON Schema 是 OpenAI 规范,DeepSeek/Kimi/智谱都兼容
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",  # 工具名,要和函数名对应
            "description": "查询某个城市的当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名,比如:北京、深圳"
                    }
                },
                "required": ["city"]  # 必填参数
            }
        }
    }
]

# Step 3: 让 LLM 处理用户问题
messages = [
    {"role": "user", "content": "深圳现在天气怎么样?"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools,  # 关键:把工具描述传给 LLM
)

# Step 4: 看 LLM 决定了什么
assistant_message = response.choices[0].message
print("--- LLM 的决定 ---")
print(f"content: {assistant_message.content}")
print(f"tool_calls: {assistant_message.tool_calls}")

--- LLM 的决定 ---
content: 好的，我来查询一下深圳现在的天气情况。
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_00_ki8MEcj6oYn2QSE2Oxoz3694', function=Function(arguments='{"city": "深圳"}', name='get_weather'), type='function', index=0)]


In [8]:
# Step 5: 解析 LLM 的工具调用决定
tool_call = assistant_message.tool_calls[0]
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)  # arguments 是 JSON 字符串,要解析

print(f"LLM 决定调用: {tool_name}")
print(f"参数: {tool_args}")

# Step 6: 你的代码真的去执行这个工具
if tool_name == "get_weather":
    tool_result = get_weather(**tool_args)  # 等价于 get_weather(city="深圳")
    print(f"工具返回: {tool_result}")

# Step 7: 把工具结果加回 messages,让 LLM 再次思考
messages.append(assistant_message)  # 把 LLM 的"调用决定"加进去
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # 标识这是对哪次工具调用的回复
    "content": tool_result
})

# Step 8: 让 LLM 用自然语言回复用户
final_response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools,  # 工具描述要一直带着
)

print("\n--- 最终回复 ---")
print(final_response.choices[0].message.content)

LLM 决定调用: get_weather
参数: {'city': '深圳'}
工具返回: 26 度,晴

--- 最终回复 ---
深圳目前的天气情况如下：

- **温度**：26°C
- **天气状况**：☀️ 晴天

整体来说是个温暖舒适的晴天，非常适合外出活动哦！不过出门的话还是建议做好防晒措施。😊


In [9]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
import json

# Step 1: 定义 State —— 我们的 Agent 需要记住什么
class AgentState(TypedDict):
    # messages 是对话历史。
    # Annotated[..., add_messages] 是 LangGraph 的特殊语法:
    # 告诉 LangGraph "更新 messages 时用追加,不要覆盖"
    messages: Annotated[list, add_messages]

# Step 2: 定义工具(就是上次那个 get_weather)
def get_weather(city: str) -> str:
    """查询某个城市的当前天气"""
    fake_data = {
        "北京": "15 度,多云",
        "深圳": "26 度,晴",
        "上海": "20 度,小雨",
    }
    return fake_data.get(city, f"{city} 的天气查不到")

# Step 3: 初始化 LLM,并把工具"绑定"给它
# bind_tools 是 LangChain 的语法,等价于上次手写的 tools=[{...}]
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    http_client=httpx.Client(trust_env=False),
)

# 把 Python 函数转成 LangChain 的 tool 对象
from langchain_core.tools import tool

@tool
def get_weather_tool(city: str) -> str:
    """查询某个城市的当前天气"""
    return get_weather(city)

llm_with_tools = llm.bind_tools([get_weather_tool])

# Step 4: 定义两个 Node
def agent_node(state: AgentState):
    """LLM 思考节点:决定要调工具还是直接回复"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def tool_node(state: AgentState):
    """工具执行节点:真的去调工具"""
    last_message = state["messages"][-1]
    tool_results = []
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "get_weather_tool":
            result = get_weather(**tool_call["args"])
            tool_results.append(ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            ))
    return {"messages": tool_results}

# Step 5: 定义 Conditional Edge —— LLM 之后该往哪走?
def should_continue(state: AgentState) -> str:
    """根据 LLM 的输出决定下一步"""
    last_message = state["messages"][-1]
    # 如果 LLM 决定调工具,去 tool_node
    if last_message.tool_calls:
        return "tools"
    # 否则结束,把回复给用户
    return END

# Step 6: 用 StateGraph 把节点和边拼起来
graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")             # 入口 → agent
graph.add_conditional_edges(
    "agent",
    should_continue,                        # 用这个函数判断下一步
    {"tools": "tools", END: END}            # 映射:返回 "tools" 走到 tools 节点,返回 END 就结束
)
graph.add_edge("tools", "agent")            # 工具执行完回到 agent(关键的循环边!)

# 编译成可执行的 Agent
app = graph.compile()

print("Agent 构建完成")



langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


Agent 构建完成


In [10]:
# 给 Agent 一个问题,看它自己走流程
result = app.invoke({
    "messages": [HumanMessage(content="深圳现在天气怎么样?")]
})

# 看每一步发生了什么
for msg in result["messages"]:
    print(f"--- {type(msg).__name__} ---")
    print(msg.content if msg.content else f"(tool_calls: {msg.tool_calls})")
    print()

--- HumanMessage ---
深圳现在天气怎么样?

--- AIMessage ---
好的，我来帮你查询深圳当前的天气情况！

--- ToolMessage ---
26 度,晴

--- AIMessage ---
深圳现在的天气是**晴朗**，气温 **26°C**，天气不错，适合外出活动哦！

